# Homework: Search Evaluation

This notebook addresses the homework questions from the LLM Zoomcamp Module 3: Evaluation.

We will evaluate different search approaches including:
- MinSearch text search with boosting
- Vector search using TF-IDF and SVD
- Qdrant vector database
- Cosine similarity evaluation
- ROUGE score evaluation

## Setup and Installation

First, let's install the required libraries:

In [1]:
# Install required libraries
!pip install -U minsearch qdrant_client scikit-learn rouge numpy pandas tqdm requests

Defaulting to user installation because normal site-packages is not writeable
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached h2-4.2.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached hpack-4.1.0-py3-none-any.whl.metadata (4.6 kB)
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.7 MB 3.3 MB/s eta 0:00:03
   --- ------------------------------------ 0.8/8.7 MB 3.3 MB/s eta 0:00:03
   ------- -------------------------------- 1.6/8.7 MB 2.7 MB/s eta 0:00:03
   --------- ------------------------------ 2.1/8.7 MB 2.7 MB/s eta 0:00:03
   ------------- -------------------------- 2.9/8.7 MB 2.8 MB/s eta 0:00:03
   --------------- ------------------------ 3.4/8.7 MB 2.6 MB/s eta 0:00:03
   ---------------- ----------------------- 3.7/8.7 MB 2.5 MB/s eta 0:00:03
   ------------------- -------------------- 4.2/8.7 MB 2.5 MB/s eta

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-sql-connector 4.0.3 requires pandas<2.3.0,>=1.2.5; python_version >= "3.8" and python_version < "3.13", but you have pandas 2.3.1 which is incompatible.
dbt-core 1.9.4 requires dbt-semantic-interfaces<0.8,>=0.7.4, but you have dbt-semantic-interfaces 0.8.5 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Libraries

In [2]:
import requests
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# MinSearch
import minsearch
from minsearch import VectorSearch

# Scikit-learn for embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline

# ROUGE scorer
from rouge import Rouge

# Qdrant
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

## Load Evaluation Data

In [3]:
# Load documents and ground truth data
url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient='records')

print(f"Loaded {len(documents)} documents")
print(f"Loaded {len(ground_truth)} ground truth questions")
print("\nSample document:")
print(documents[0])
print("\nSample ground truth:")
print(ground_truth[0])

Loaded 948 documents
Loaded 4627 ground truth questions

Sample document:
{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.", 'section': 'General course-related questions', 'question': 'Course - When will the course start?', 'course': 'data-engineering-zoomcamp', 'id': 'c02e79ef'}

Sample ground truth:
{'question': 'When does the course begin?', 'course': 'data-engineering-zoomcamp', 'document': 'c02e79ef'}


## Evaluation Functions

In [4]:
def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if True in line:
            cnt = cnt + 1
    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)
                break  # Only count the first relevant result
    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []
    
    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)
    
    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

## Q1. MinSearch Text with Boosting

Evaluate MinSearch with text fields and boosting parameters:
- `text_fields=["question", "section", "text"]`
- `keyword_fields=["course", "id"]`
- `boost = {'question': 1.5, 'section': 0.1}`

In [5]:
# Initialize MinSearch index
index = minsearch.Index(
    text_fields=["question", "section", "text"],
    keyword_fields=["course", "id"]
)

# Fit the index with documents
index.fit(documents)

# Define search function with boosting
def minsearch_search(query_data):
    query = query_data['question']
    course = query_data['course']
    
    boost = {'question': 1.5, 'section': 0.1}
    
    results = index.search(
        query=query,
        filter_dict={'course': course},
        boost_dict=boost,
        num_results=10
    )
    
    return results

# Evaluate Q1
print("Q1. Evaluating MinSearch with boosting...")
results_q1 = evaluate(ground_truth, minsearch_search)
print(f"Hit Rate: {results_q1['hit_rate']:.4f}")
print(f"MRR: {results_q1['mrr']:.4f}")
print(f"\nQ1 Answer: Hit Rate = {results_q1['hit_rate']:.2f}")

Q1. Evaluating MinSearch with boosting...


  0%|          | 0/4627 [00:00<?, ?it/s]

Hit Rate: 0.8993
MRR: 0.7351

Q1 Answer: Hit Rate = 0.90


## Q2. Vector Search for Question Only

Create embeddings using TF-IDF and SVD for the "question" field only.

In [6]:
# Create embeddings for question field only
texts_q2 = []
for doc in documents:
    t = doc['question']
    texts_q2.append(t)

# Create pipeline for embeddings
pipeline_q2 = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

# Fit and transform
X_q2 = pipeline_q2.fit_transform(texts_q2)
print(f"Embeddings shape: {X_q2.shape}")

# Index with VectorSearch
vindex_q2 = VectorSearch(keyword_fields={'course'})
vindex_q2.fit(X_q2, documents)

# Define search function for Q2
def vector_search_q2(query_data):
    query = query_data['question']
    course = query_data['course']
    
    # Transform query to embedding
    query_vector = pipeline_q2.transform([query])
    
    # Search
    results = vindex_q2.search(
        query_vector=query_vector[0],
        filter_dict={'course': course},
        num_results=10
    )
    
    return results

# Evaluate Q2
print("\nQ2. Evaluating Vector Search (question only)...")
results_q2 = evaluate(ground_truth, vector_search_q2)
print(f"Hit Rate: {results_q2['hit_rate']:.4f}")
print(f"MRR: {results_q2['mrr']:.4f}")
print(f"\nQ2 Answer: MRR = {results_q2['mrr']:.2f}")

Embeddings shape: (948, 128)

Q2. Evaluating Vector Search (question only)...


  0%|          | 0/4627 [00:00<?, ?it/s]

Hit Rate: 0.5606
MRR: 0.3673

Q2 Answer: MRR = 0.37


## Q3. Vector Search for Question and Answer

Use both question and text (answer) fields for embeddings.

In [7]:
# Create embeddings for question + text fields
texts_q3 = []
for doc in documents:
    t = doc['question'] + ' ' + doc['text']
    texts_q3.append(t)

# Create pipeline for embeddings (same parameters as Q2)
pipeline_q3 = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

# Fit and transform
X_q3 = pipeline_q3.fit_transform(texts_q3)
print(f"Embeddings shape: {X_q3.shape}")

# Index with VectorSearch
vindex_q3 = VectorSearch(keyword_fields={'course'})
vindex_q3.fit(X_q3, documents)

# Define search function for Q3
def vector_search_q3(query_data):
    query = query_data['question']
    course = query_data['course']
    
    # Transform query to embedding
    query_vector = pipeline_q3.transform([query])
    
    # Search
    results = vindex_q3.search(
        query_vector=query_vector[0],
        filter_dict={'course': course},
        num_results=10
    )
    
    return results

# Evaluate Q3
print("\nQ3. Evaluating Vector Search (question + text)...")
results_q3 = evaluate(ground_truth, vector_search_q3)
print(f"Hit Rate: {results_q3['hit_rate']:.4f}")
print(f"MRR: {results_q3['mrr']:.4f}")
print(f"\nQ3 Answer: Hit Rate = {results_q3['hit_rate']:.2f}")

Embeddings shape: (948, 128)

Q3. Evaluating Vector Search (question + text)...


  0%|          | 0/4627 [00:00<?, ?it/s]

Hit Rate: 0.8842
MRR: 0.6800

Q3 Answer: Hit Rate = 0.88


## Q4. Qdrant Vector Database

Evaluate using Qdrant with:
- `text = doc['question'] + ' ' + doc['text']`
- `model_handle = "jinaai/jina-embeddings-v2-small-en"`
- `limit = 5`

In [8]:
# Initialize Qdrant client (in-memory)
client = QdrantClient(":memory:")

# Initialize the sentence transformer model
model_name = "jinaai/jina-embeddings-v2-small-en"
model = SentenceTransformer(model_name)

print(f"Model embedding dimension: {model.get_sentence_embedding_dimension()}")

# Create collection
collection_name = "homework_collection"
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=model.get_sentence_embedding_dimension(),
        distance=Distance.COSINE,
    ),
)

# Prepare and upload documents
print("Creating embeddings and uploading to Qdrant...")
points = []
for i, doc in enumerate(tqdm(documents)):
    text = doc['question'] + ' ' + doc['text']
    vector = model.encode(text)
    
    points.append(PointStruct(
        id=i,
        vector=vector.tolist(),
        payload=doc
    ))

# Upload points in batches
batch_size = 100
for i in range(0, len(points), batch_size):
    batch = points[i:i+batch_size]
    client.upsert(collection_name=collection_name, points=batch)

print(f"Uploaded {len(points)} documents to Qdrant")

# Define search function for Q4
def qdrant_search_q4(query_data):
    query = query_data['question']
    course = query_data['course']
    
    # Create query embedding
    query_vector = model.encode(query)
    
    # Search with course filter
    search_results = client.search(
        collection_name=collection_name,
        query_vector=query_vector.tolist(),
        query_filter={
            "must": [
                {
                    "key": "course",
                    "match": {
                        "value": course
                    }
                }
            ]
        },
        limit=5  # As specified in Q4
    )
    
    # Convert to the expected format
    results = []
    for hit in search_results:
        results.append(hit.payload)
    
    return results

# Evaluate Q4
print("\nQ4. Evaluating Qdrant Vector Search...")
results_q4 = evaluate(ground_truth, qdrant_search_q4)
print(f"Hit Rate: {results_q4['hit_rate']:.4f}")
print(f"MRR: {results_q4['mrr']:.4f}")
print(f"\nQ4 Answer: MRR = {results_q4['mrr']:.2f}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\TamKC\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\TamKC\.cache\huggingface\hub\models--jinaai--jina-embeddings-v2-small-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

SSLError: (MaxRetryError("HTTPSConnectionPool(host='cdn-lfs.hf.co', port=443): Max retries exceeded with url: /repos/bd/0f/bd0f461f82064b108f7caf5fbd0fa9dd0fcedd1d022494bcb69358d97b09056a/c9a9a7ec012d01efd780474fbb65e25917f3a2aebdff84b5f87daa00f7e90b27?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1755095238&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc1NTA5NTIzOH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy9iZC8wZi9iZDBmNDYxZjgyMDY0YjEwOGY3Y2FmNWZiZDBmYTlkZDBmY2VkZDFkMDIyNDk0YmNiNjkzNThkOTdiMDkwNTZhL2M5YTlhN2VjMDEyZDAxZWZkNzgwNDc0ZmJiNjVlMjU5MTdmM2EyYWViZGZmODRiNWY4N2RhYTAwZjdlOTBiMjc~cmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIn1dfQ__&Signature=Kdl7KBd~N-qNYTpyk9PQfvDgc1uj3GfB7~B6QCwOrfyM7X9oqNdeo018EOTyyM15-8oSGhKWBtScbubJTpePsCoswzvYj2miQy-u1nJVyIeF9QNyKgwHbhJCLjhQFqkjUhCVjCNVGSbmsmrrwtjahHeQ2tJpYV86rRnTz0I2Nko80sipvji1KJEYVwnNj2UlI0X2rtVH7YlZNG~VA

## Q5. Cosine Similarity

Calculate cosine similarity between LLM-generated and original answers.

In [9]:
# Load evaluation results
results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)

print(f"Loaded {len(df_results)} evaluation results")
print("\nSample result:")
print(df_results.iloc[0])

# Define cosine similarity functions
def normalize(u):
    norm = np.sqrt(u.dot(u))
    return u / norm

def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    return u.dot(v) / (u_norm * v_norm)

# Alternative simplified version
def cosine_simple(u, v):
    u = normalize(u)
    v = normalize(v)
    return u.dot(v)

# Create pipeline for cosine similarity
pipeline_cosine = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)

# Fit on all text data
all_texts = df_results.answer_llm + ' ' + df_results.answer_orig + ' ' + df_results.question
pipeline_cosine.fit(all_texts)

print("\nCalculating cosine similarities...")
cosine_similarities = []

for idx, row in tqdm(df_results.iterrows(), total=len(df_results)):
    # Create embeddings for LLM and original answers
    v_llm = pipeline_cosine.transform([row.answer_llm])[0]
    v_orig = pipeline_cosine.transform([row.answer_orig])[0]
    
    # Calculate cosine similarity
    similarity = cosine(v_llm, v_orig)
    cosine_similarities.append(similarity)

# Calculate average cosine similarity
avg_cosine = np.mean(cosine_similarities)

print(f"\nAverage Cosine Similarity: {avg_cosine:.4f}")
print(f"Q5 Answer: Average Cosine = {avg_cosine:.2f}")

# Show distribution
print(f"\nCosine Similarity Statistics:")
print(f"Min: {np.min(cosine_similarities):.4f}")
print(f"Max: {np.max(cosine_similarities):.4f}")
print(f"Std: {np.std(cosine_similarities):.4f}")

Loaded 1830 evaluation results

Sample result:
answer_llm     You can sign up for the course by visiting the...
answer_orig    Machine Learning Zoomcamp FAQ\nThe purpose of ...
document                                                0227b872
question                     Where can I sign up for the course?
course                                 machine-learning-zoomcamp
Name: 0, dtype: object

Calculating cosine similarities...

Calculating cosine similarities...


  0%|          | 0/1830 [00:00<?, ?it/s]


Average Cosine Similarity: 0.8416
Q5 Answer: Average Cosine = 0.84

Cosine Similarity Statistics:
Min: 0.0791
Max: 0.9965
Std: 0.1737


## Q6. ROUGE Score

Calculate ROUGE-1 F1 scores for answer pairs.

In [12]:
# Check if we have the required columns
print("Checking dataframe columns...")
print(f"DataFrame columns: {list(df_results.columns)}")
print(f"DataFrame shape: {df_results.shape}")

# Initialize ROUGE scorer
try:
    rouge_scorer = Rouge()
    print("ROUGE scorer initialized successfully")
except Exception as e:
    print(f"Error initializing ROUGE: {e}")
    print("Installing ROUGE...")
    import subprocess
    subprocess.run(["pip", "install", "rouge"], check=True)
    from rouge import Rouge
    rouge_scorer = Rouge()

# Helper function to clean text for ROUGE
def clean_text_for_rouge(text):
    """Clean text to avoid ROUGE errors"""
    if pd.isna(text) or text is None:
        return "no answer"
    
    text = str(text).strip()
    if len(text) == 0:
        return "no answer"
    
    # Remove extra whitespace and ensure we have at least some content
    text = ' '.join(text.split())
    if len(text) == 0:
        return "no answer"
    
    return text

# Test with the 10th document as mentioned in the homework
r = df_results.iloc[10]
print(f"\nDocument at index 10:")
print(f"LLM Answer: {r.answer_llm[:100]}...")
print(f"Original Answer: {r.answer_orig[:100]}...")

# Clean the texts
llm_answer_clean = clean_text_for_rouge(r.answer_llm)
orig_answer_clean = clean_text_for_rouge(r.answer_orig)

# Calculate ROUGE scores for the 10th document
try:
    scores = rouge_scorer.get_scores(llm_answer_clean, orig_answer_clean)[0]
    print(f"\nROUGE scores for document 10:")
    print(scores)
    print(f"\nROUGE-1 F1 for document 10: {scores['rouge-1']['f']:.4f}")
except Exception as e:
    print(f"Error calculating ROUGE for document 10: {e}")

# Calculate ROUGE-1 F1 for all pairs
print("\nCalculating ROUGE-1 F1 for all pairs...")
rouge1_f1_scores = []

for idx, row in tqdm(df_results.iterrows(), total=len(df_results)):
    try:
        # Clean the texts
        llm_answer = clean_text_for_rouge(row.answer_llm)
        orig_answer = clean_text_for_rouge(row.answer_orig)
        
        # Calculate ROUGE scores
        scores = rouge_scorer.get_scores(llm_answer, orig_answer)[0]
        rouge1_f1 = scores['rouge-1']['f']
        rouge1_f1_scores.append(rouge1_f1)
        
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        print(f"LLM answer: {row.answer_llm[:50] if pd.notna(row.answer_llm) else 'None'}")
        print(f"Original answer: {row.answer_orig[:50] if pd.notna(row.answer_orig) else 'None'}")
        rouge1_f1_scores.append(0.0)

# Calculate average ROUGE-1 F1
avg_rouge1_f1 = np.mean(rouge1_f1_scores)

print(f"\nROUGE-1 F1 Results:")
print(f"Total pairs processed: {len(rouge1_f1_scores)}")
print(f"Average ROUGE-1 F1: {avg_rouge1_f1:.4f}")
print(f"Q6 Answer: Average ROUGE-1 F1 = {avg_rouge1_f1:.2f}")

# Show distribution
print(f"\nROUGE-1 F1 Statistics:")
print(f"Min: {np.min(rouge1_f1_scores):.4f}")
print(f"Max: {np.max(rouge1_f1_scores):.4f}")
print(f"Std: {np.std(rouge1_f1_scores):.4f}")

# Show some sample scores
print(f"\nFirst 10 ROUGE-1 F1 scores:")
for i in range(min(10, len(rouge1_f1_scores))):
    print(f"Row {i}: {rouge1_f1_scores[i]:.4f}")

Checking dataframe columns...
DataFrame columns: ['answer_llm', 'answer_orig', 'document', 'question', 'course']
DataFrame shape: (1830, 5)
ROUGE scorer initialized successfully

Document at index 10:
LLM Answer: Yes, all sessions are recorded, so if you miss one, you won't miss anything. You can catch up on the...
Original Answer: Everything is recorded, so you won’t miss anything. You will be able to ask your questions for offic...

ROUGE scores for document 10:
{'rouge-1': {'r': 0.45454545454545453, 'p': 0.45454545454545453, 'f': 0.45454544954545456}, 'rouge-2': {'r': 0.21621621621621623, 'p': 0.21621621621621623, 'f': 0.21621621121621637}, 'rouge-l': {'r': 0.3939393939393939, 'p': 0.3939393939393939, 'f': 0.393939388939394}}

ROUGE-1 F1 for document 10: 0.4545

Calculating ROUGE-1 F1 for all pairs...


  0%|          | 0/1830 [00:00<?, ?it/s]


ROUGE-1 F1 Results:
Total pairs processed: 1830
Average ROUGE-1 F1: 0.3517
Q6 Answer: Average ROUGE-1 F1 = 0.35

ROUGE-1 F1 Statistics:
Min: 0.0000
Max: 0.9500
Std: 0.1589

First 10 ROUGE-1 F1 scores:
Row 0: 0.0952
Row 1: 0.1250
Row 2: 0.4156
Row 3: 0.2162
Row 4: 0.1421
Row 5: 0.4314
Row 6: 0.4127
Row 7: 0.3043
Row 8: 0.5172
Row 9: 0.3437


## Summary of Results

Let's summarize all the answers:

In [ ]:
print("=" * 60)
print("HOMEWORK RESULTS SUMMARY")
print("=" * 60)

print(f"Q1. MinSearch with boosting - Hit Rate: {results_q1['hit_rate']:.2f}")
print(f"Q2. Vector Search (question) - MRR: {results_q2['mrr']:.2f}")
print(f"Q3. Vector Search (question+text) - Hit Rate: {results_q3['hit_rate']:.2f}")
print(f"Q4. Qdrant Vector Search - MRR: {results_q4['mrr']:.2f}")
print(f"Q5. Cosine Similarity - Average: {avg_cosine:.2f}")
print(f"Q6. ROUGE-1 F1 - Average: {avg_rouge1_f1:.2f}")

print("\n" + "=" * 60)
print("Select the closest answer from the multiple choice options!")
print("=" * 60)

## Additional Analysis

Let's compare the different approaches:

In [ ]:
# Create comparison table
comparison_data = {
    'Method': [
        'MinSearch (boosted)',
        'Vector (question only)',
        'Vector (question+text)',
        'Qdrant'
    ],
    'Hit Rate': [
        results_q1['hit_rate'],
        results_q2['hit_rate'],
        results_q3['hit_rate'],
        results_q4['hit_rate']
    ],
    'MRR': [
        results_q1['mrr'],
        results_q2['mrr'],
        results_q3['mrr'],
        results_q4['mrr']
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("Search Method Comparison:")
print(df_comparison.round(4))

print(f"\nEvaluation Method Comparison:")
print(f"Cosine Similarity: {avg_cosine:.4f}")
print(f"ROUGE-1 F1: {avg_rouge1_f1:.4f}")